# Tugas 1 — Milestone 3: Analisis Temporal, Spasial, dan Advanced EDA

**Mata kuliah:** Analisis Big Data  
**Nama:** Dino Ibnu Zein  
**NIM:** 202310370311107  
**GitHub:** dinoibnuzein

Notebook melanjutkan data Transjakarta pada Milestone 1 dan hasil cleaning
Milestone 2, dengan enam visualisasi interaktif serta enam insight yang dihitung
dari hasil analisis. Periode aktual adalah **26 November 2019, 14.00–17.59**.
Pengamatan empat jam ini belum cukup untuk menyimpulkan pola harian berulang
atau musiman. Timestamp dibaca sesuai sumber, tanpa menambahkan zona waktu.
Speed menggunakan satuan pada dataset, yang belum dikonfirmasi dalam metadata lokal.

**Menjalankan:** pasang `requirements.txt`, buka notebook dari root proyek atau
folder `notebooks/`, lalu pilih **Run All**. Inputnya adalah
`data/processed/transjakarta_gps_bersih.parquet` dan audit di
`output/milestone2/ringkasan_cleaning.json`.
Hasil interaktif mandiri tersedia di `output/milestone3/ringkasan_interaktif.html`.

## Bagian 1: Inisialisasi dan Memuat Data Bersih

Input bersih dan audit M2 dibaca tanpa mengubah file data atau notebook M1–2.
Semua hasil analisis disimpan pada folder output Milestone 3.

In [13]:
from pathlib import Path
import hashlib
import html
import json
import platform
import duckdb
import polars as pl
import plotly
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import display, Markdown

def find_project_root():
    here=Path.cwd().resolve()
    for p in (here,*here.parents):
        if (p/'data/processed/transjakarta_gps_bersih.parquet').is_file():return p
    raise FileNotFoundError('Jalankan dari root proyek atau notebooks/; Parquet hasil M2 harus tersedia.')

PROJECT_ROOT=find_project_root()
DATA_PATH=PROJECT_ROOT/'data/processed/transjakarta_gps_bersih.parquet'
OUTPUT_DIR=PROJECT_ROOT/'output/milestone3'
FIGURE_DIR=PROJECT_ROOT/'output/figures/milestone3'
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
FIGURE_DIR.mkdir(parents=True,exist_ok=True)
INPUT_SHA256=hashlib.sha256(DATA_PATH.read_bytes()).hexdigest()
audit_m2=json.loads((PROJECT_ROOT/'output/milestone2/ringkasan_cleaning.json').read_text())
pl.Config.set_tbl_rows(12)
pl.Config.set_tbl_cols(10)
pio.templates.default='plotly_white'
pio.renderers.default='plotly_mimetype'
CONFIG={'displaylogo':False,'responsive':True,'toImageButtonOptions':{'format':'png','scale':2}}
COLORS={'navy':'#15324F','teal':'#087F8C','orange':'#D97732','gray':'#7D8B99'}
figures=[]
figure_files=[]

def publish(fig,name,height=540):
    fig.update_layout(font={'family':'Arial, sans-serif','size':12,'color':'#21364B'},
        height=height,margin={'l':80,'r':70,'t':90,'b':95},
        paper_bgcolor='white',plot_bgcolor='#F7FAFC',title={'x':0.02})
    fig.write_html(FIGURE_DIR/name,include_plotlyjs='directory',full_html=True,
                   config=CONFIG,auto_open=False)
    figures.append(fig);figure_files.append(name)
    fig.show(config=CONFIG)

versions={'python':platform.python_version(),'polars':pl.__version__,
          'duckdb':duckdb.__version__,'plotly':plotly.__version__}
print('Versi analitik:',versions)
print('Input:',DATA_PATH.relative_to(PROJECT_ROOT).as_posix())

Versi analitik: {'python': '3.13.0', 'polars': '1.44.2', 'duckdb': '1.5.6', 'plotly': '7.1.0'}
Input: data/processed/transjakarta_gps_bersih.parquet


### Pemeriksaan input dan hubungan dengan M2

Audit M2 mencatat 775.458 baris awal, 48.384 duplikat disisihkan, dan 727.074
baris bersih. Jumlah baris diperiksa terhadap file aktual. Speed null tetap
berarti tidak diketahui, sedangkan nol adalah nilai pengamatan.

Notebook M2 pada arsip yang diberikan masih berupa template. M3 menggunakan
Parquet bersih dan audit yang tersedia; belum dapat diklaim bahwa template M2
tersebut mampu menghasilkan ulang cleaning dari data mentah.

In [14]:
df=pl.scan_parquet(DATA_PATH).collect()
required={'bus_code','gps_datetime','corridor','speed','longitude','latitude','outside_jabodetabek_check'}
assert required<=set(df.columns)
assert df.schema['gps_datetime'].base_type()==pl.Datetime
checks_input=pl.DataFrame([
    {'pemeriksaan':'Bus null','pelanggaran':df['bus_code'].null_count()},
    {'pemeriksaan':'Timestamp null','pelanggaran':df['gps_datetime'].null_count()},
    {'pemeriksaan':'Duplikat bus-timestamp','pelanggaran':df.height-df.select('bus_code','gps_datetime').unique().height},
    {'pemeriksaan':'Speed non-null di luar [0,120]','pelanggaran':df.select((pl.col('speed').is_not_null() & ~pl.col('speed').is_between(0,120)).sum()).item()},
    {'pemeriksaan':'Baris berbeda dari audit M2','pelanggaran':int(df.height!=audit_m2['baris_bersih'])},
])
assert checks_input['pelanggaran'].sum()==0
start_time=df['gps_datetime'].min();end_time=df['gps_datetime'].max()
valid_speed=df['speed'].count();null_speed=df['speed'].null_count()
zero_speed=df.select(pl.col('speed').eq(0).sum()).item()
low_speed=df.select(pl.col('speed').le(5).sum()).item()
outside_count=df['outside_jabodetabek_check'].sum()
input_summary=pl.DataFrame({'indikator':['Baris bersih','Bus unik eksak','Kode koridor unik','Speed null','Speed nol','Penanda luar wilayah M2'],
    'nilai':[df.height,df['bus_code'].n_unique(),df['corridor'].n_unique(),null_speed,zero_speed,outside_count]})
print('Periode:',start_time,'sampai',end_time)
display(input_summary);display(checks_input)
display(df.select('bus_code','gps_datetime','corridor','longitude','latitude','speed').head())
input_summary.write_csv(OUTPUT_DIR/'ringkasan_input.csv')

Periode: 2019-11-26 14:00:00 sampai 2019-11-26 17:59:59


indikator,nilai
str,i64
"""Baris bersih""",727074
"""Bus unik eksak""",1783
"""Kode koridor unik""",127
"""Speed null""",99
"""Speed nol""",274670
"""Penanda luar wilayah M2""",2


pemeriksaan,pelanggaran
str,i64
"""Bus null""",0
"""Timestamp null""",0
"""Duplikat bus-timestamp""",0
"""Speed non-null di luar [0,120]""",0
"""Baris berbeda dari audit M2""",0


bus_code,gps_datetime,corridor,longitude,latitude,speed
str,datetime[μs],str,f64,f64,f64
"""DMR 5049""",2019-11-26 14:00:29,"""0""",106.867523,-6.22423,0.0
"""DMR 5049""",2019-11-26 14:00:54,"""0""",106.867523,-6.22423,0.0
"""DMR 5049""",2019-11-26 14:01:29,"""0""",106.867523,-6.22423,0.0
"""DMR 5049""",2019-11-26 14:01:59,"""0""",106.867523,-6.22423,0.0
"""DMR 5049""",2019-11-26 14:02:29,"""0""",106.867523,-6.22423,0.0


## Bagian 2: Analisis Temporal dan Window Functions

`group_by_dynamic(every='5m')` menggunakan interval [awal, akhir) agar rekaman
tidak terhitung dua kali. Rata-rata per rekaman dibandingkan dengan rata-rata
per bus: bus dengan speed valid diberi bobot sama dalam setiap interval.
`rolling_mean(3)` menggunakan tiga interval penuh (15 menit); dua titik awal null.
`shift()` dibandingkan dengan DuckDB `LAG`, dan moving average diperiksa
menggunakan SQL window `AVG`. Interval kosong diperiksa agar MA tidak salah
diartikan sebagai durasi 15 menit.

Bus tercatat melapor bukan ukuran jumlah penumpang atau kepastian sedang beroperasi.

In [15]:
record_windows=(df.sort('gps_datetime')
    .group_by_dynamic('gps_datetime',every='5m',closed='left',label='left')
    .agg(pl.len().alias('jumlah_rekaman'),pl.col('bus_code').n_unique().alias('bus_tercatat'),
         pl.col('speed').count().alias('speed_valid'),pl.col('speed').mean().alias('speed_mean_rekaman'),
         (pl.col('speed').eq(0).sum()/pl.col('speed').count()*100).alias('persen_speed_nol')))
df_sorted=df.sort(['bus_code','gps_datetime'])
bus_windows=(df_sorted.group_by_dynamic('gps_datetime',every='5m',group_by='bus_code',closed='left',label='left')
    .agg(pl.col('speed').mean().alias('speed_mean_per_bus')))
balanced_windows=bus_windows.group_by('gps_datetime').agg(
    pl.col('speed_mean_per_bus').mean().alias('speed_mean_bus'),
    pl.col('speed_mean_per_bus').count().alias('bus_dengan_speed_valid'))
temporal=(record_windows.join(balanced_windows,on='gps_datetime',how='left').sort('gps_datetime')
    .with_columns(pl.col('speed_mean_bus').rolling_mean(3).alias('ma_15_menit'),
                  pl.col('speed_mean_bus').shift(1).alias('speed_interval_sebelumnya'))
    .with_columns((pl.col('speed_mean_bus')-pl.col('speed_interval_sebelumnya')).alias('perubahan_speed')))
assert temporal['jumlah_rekaman'].sum()==df.height
assert temporal['gps_datetime'].diff().drop_nulls().dt.total_seconds().eq(300).all()
con=duckdb.connect();con.register('temporal_polars',temporal.to_arrow())
sql_result=con.execute("""
SELECT gps_datetime,LAG(speed_mean_bus) OVER(ORDER BY gps_datetime) AS lag_sql,
       CASE WHEN COUNT(speed_mean_bus) OVER w=3 THEN AVG(speed_mean_bus) OVER w END AS ma_sql,
       SUM(jumlah_rekaman) OVER(ORDER BY gps_datetime) AS rekaman_kumulatif
FROM temporal_polars
WINDOW w AS(ORDER BY gps_datetime ROWS BETWEEN 2 PRECEDING AND CURRENT ROW)
ORDER BY gps_datetime
""")
arrow=(sql_result.to_arrow_table() if hasattr(sql_result,'to_arrow_table') else sql_result.fetch_arrow_table())
sql_window=pl.from_arrow(arrow)
comparison=temporal.join(sql_window,on='gps_datetime',how='left')
ma_error=comparison.select((pl.col('ma_15_menit')-pl.col('ma_sql')).abs().max()).item()
lag_error=comparison.select((pl.col('speed_interval_sebelumnya')-pl.col('lag_sql')).abs().max()).item()
assert ma_error<1e-9 and lag_error<1e-9
assert int(sql_window['rekaman_kumulatif'][-1])==df.height
temporal.write_csv(OUTPUT_DIR/'temporal_5_menit.csv')
sql_window.write_csv(OUTPUT_DIR/'window_duckdb.csv')
print('Interval:',temporal.height,'; selisih MA Polars-DuckDB:',ma_error)
display(temporal.head(8))

Interval: 48 ; selisih MA Polars-DuckDB: 3.552713678800501e-15


gps_datetime,jumlah_rekaman,bus_tercatat,speed_valid,speed_mean_rekaman,…,speed_mean_bus,bus_dengan_speed_valid,ma_15_menit,speed_interval_sebelumnya,perubahan_speed
datetime[μs],u32,u32,u32,f64,…,f64,u32,f64,f64,f64
2019-11-26 14:00:00,14122,1540,14122,10.233154,…,10.102886,1540,null,null,null
2019-11-26 14:05:00,14702,1550,14702,9.88592,…,9.803096,1550,null,10.102886,-0.29979
2019-11-26 14:10:00,14829,1552,14829,9.72133,…,9.649832,1552,9.851938,9.803096,-0.153264
2019-11-26 14:15:00,14972,1563,14972,9.849693,…,9.731985,1563,9.728304,9.649832,0.082153
2019-11-26 14:20:00,15153,1577,15153,10.603234,…,10.445272,1577,9.942363,9.731985,0.713287
2019-11-26 14:25:00,15250,1593,15250,10.704846,…,10.632134,1593,10.269797,10.445272,0.186862
2019-11-26 14:30:00,15271,1591,15271,10.330109,…,10.259225,1591,10.445544,10.632134,-0.372909
2019-11-26 14:35:00,15376,1597,15366,10.327463,…,10.233352,1596,10.374904,10.259225,-0.025873


### Agregasi bus, distribusi, koridor, jam, dan grid spasial

Perhitungan memakai seluruh data bersih. Scatter plot memakai bus dengan minimal
30 rekaman dan speed valid (ambang eksplorasi, bukan standar operasional).
Kode koridor `0` tetap dilaporkan tetapi tidak diasumsikan sebagai satu rute
operasional. Grafik spasial memakai sel 0,005° (sekitar 550 m), minimal 30
rekaman per sel, serta memisahkan penanda luar wilayah M2 sebagai tabel audit.

In [16]:
with_gaps=df_sorted.with_columns(pl.col('gps_datetime').diff().dt.total_seconds().over('bus_code').alias('gap_detik'))
bus_stats=(with_gaps.group_by('bus_code').agg(pl.len().alias('jumlah_rekaman'),
    pl.col('speed').count().alias('speed_valid'),pl.col('speed').mean().alias('speed_mean'),
    pl.col('gps_datetime').min().alias('awal'),pl.col('gps_datetime').max().alias('akhir'),
    pl.col('gap_detik').filter(pl.col('gap_detik')>0).median().alias('median_gap_detik'),
    (pl.col('speed').eq(0).sum()/pl.col('speed').count()*100).alias('persen_speed_nol')).sort('bus_code'))
scatter_bus=bus_stats.filter((pl.col('jumlah_rekaman')>=30)&pl.col('speed_mean').is_not_null())
pearson_r=scatter_bus.select(pl.corr('jumlah_rekaman','speed_mean')).item()
speed_groups=['0','(0, 5]','(5, 10]','(10, 20]','(20, 40]','(40, 60]','(60, 120]','Tidak diketahui']
distribution=(df.select(pl.when(pl.col('speed').is_null()).then(pl.lit('Tidak diketahui'))
    .when(pl.col('speed')==0).then(pl.lit('0')).when(pl.col('speed')<=5).then(pl.lit('(0, 5]'))
    .when(pl.col('speed')<=10).then(pl.lit('(5, 10]')).when(pl.col('speed')<=20).then(pl.lit('(10, 20]'))
    .when(pl.col('speed')<=40).then(pl.lit('(20, 40]')).when(pl.col('speed')<=60).then(pl.lit('(40, 60]'))
    .otherwise(pl.lit('(60, 120]')).alias('kelompok_speed')).group_by('kelompok_speed').len(name='jumlah_rekaman'))
distribution=(pl.DataFrame({'kelompok_speed':speed_groups}).join(distribution,on='kelompok_speed',how='left').fill_null(0)
    .with_columns((pl.col('jumlah_rekaman')/df.height*100).alias('persen_seluruh_rekaman')))
assert distribution['jumlah_rekaman'].sum()==df.height
corridor_bus=df.group_by('corridor','bus_code').agg(pl.col('speed').mean().alias('speed_bus'))
corridor_stats=(df.group_by('corridor').agg(pl.len().alias('jumlah_rekaman'),pl.col('bus_code').n_unique().alias('bus_unik'),
    pl.col('speed').mean().alias('speed_mean_rekaman'))
    .join(corridor_bus.group_by('corridor').agg(pl.col('speed_bus').mean().alias('speed_mean_bus')),on='corridor',how='left')
    .sort(['jumlah_rekaman','corridor'],descending=[True,False]))
top_corridors=corridor_stats.filter(pl.col('corridor')!='0').head(12)
top_codes=top_corridors['corridor'].head(8).to_list()
bus_hours=(df_sorted.group_by_dynamic('gps_datetime',every='1h',group_by='bus_code',closed='left',label='left')
    .agg(pl.col('speed').mean().alias('speed_bus_jam')))
hourly=(df.with_columns(pl.col('gps_datetime').dt.truncate('1h').alias('jam')).group_by('jam')
    .agg(pl.len().alias('jumlah_rekaman'),pl.col('bus_code').n_unique().alias('bus_tercatat'),pl.col('speed').mean().alias('speed_mean_rekaman'))
    .join(bus_hours.group_by('gps_datetime').agg(pl.col('speed_bus_jam').mean().alias('speed_mean_bus'))
          .rename({'gps_datetime':'jam'}),on='jam',how='left').sort('jam'))
corridor_bus_hour=(df.with_columns(pl.col('gps_datetime').dt.truncate('1h').alias('jam'))
    .group_by('jam','corridor','bus_code').agg(pl.len().alias('jumlah_rekaman'),pl.col('speed').mean().alias('speed_bus_jam')))
corridor_hour=(corridor_bus_hour.group_by('jam','corridor').agg(pl.col('jumlah_rekaman').sum(),
    pl.len().alias('bus_tercatat'),pl.col('speed_bus_jam').count().alias('bus_dengan_speed_valid'),
    pl.col('speed_bus_jam').mean().alias('speed_mean_bus')).sort(['corridor','jam']))
GRID=0.005
spatial_audit=df.filter(pl.col('outside_jabodetabek_check'))
spatial_grid=(df.filter(~pl.col('outside_jabodetabek_check'))
    .with_columns((pl.col('longitude')/GRID).floor().cast(pl.Int32).alias('grid_lon'),
                  (pl.col('latitude')/GRID).floor().cast(pl.Int32).alias('grid_lat'))
    .group_by('grid_lon','grid_lat').agg(pl.len().alias('jumlah_rekaman'),pl.col('bus_code').n_unique().alias('bus_unik'),
        pl.col('speed').count().alias('speed_valid'),pl.col('speed').median().alias('speed_median'))
    .with_columns(((pl.col('grid_lon')+0.5)*GRID).alias('longitude'),((pl.col('grid_lat')+0.5)*GRID).alias('latitude'))
    .sort(['jumlah_rekaman','grid_lon','grid_lat'],descending=[True,False,False]))
spatial_plot=spatial_grid.filter((pl.col('jumlah_rekaman')>=30)&(pl.col('speed_valid')>0))
assert spatial_grid['jumlah_rekaman'].sum()+spatial_audit.height==df.height
tables={'statistik_per_bus':bus_stats,'distribusi_speed':distribution,'statistik_koridor':corridor_stats,
    'statistik_per_jam':hourly,'koridor_per_jam':corridor_hour,'grid_spasial':spatial_grid,'audit_titik_luar_wilayah':spatial_audit}
for name,table in tables.items():table.write_csv(OUTPUT_DIR/f'{name}.csv')
print('Bus pada scatter:',scatter_bus.height,'; Pearson r:',round(pearson_r,4))
display(hourly)

Bus pada scatter: 1759 ; Pearson r: 0.0995


jam,jumlah_rekaman,bus_tercatat,speed_mean_rekaman,speed_mean_bus
datetime[μs],u32,u32,f64,f64
2019-11-26 14:00:00,181021,1674,10.231116,9.989058
2019-11-26 15:00:00,163614,1732,10.661822,10.516212
2019-11-26 16:00:00,192491,1740,11.242988,11.1209
2019-11-26 17:00:00,189948,1723,10.756871,10.740049


## Bagian 3: Enam Visualisasi Interaktif

Hover untuk melihat nilai, klik legenda untuk memilih seri, dan gunakan zoom.
HTML setiap grafik memakai JavaScript lokal dalam folder yang sama. Laporan
gabungan menanam JavaScript satu kali sehingga dapat dibuka tanpa internet.

### 1. Tren speed per lima menit
Rata-rata per bus, rata-rata per rekaman, dan MA 15 menit dibandingkan. Perbedaan
bobot pelaporan tidak disamakan dengan perubahan jumlah penumpang.

In [17]:
fig1=go.Figure()
hover=[[int(r['jumlah_rekaman']),int(r['bus_tercatat'])] for r in temporal.iter_rows(named=True)]
for col,label,color,dash in [('speed_mean_bus','Rata-rata per bus',COLORS['teal'],'solid'),
    ('speed_mean_rekaman','Rata-rata per rekaman',COLORS['gray'],'dot'),
    ('ma_15_menit','Moving average 15 menit',COLORS['navy'],'dash')]:
    fig1.add_trace(go.Scatter(x=temporal['gps_datetime'].to_list(),y=temporal[col].to_list(),name=label,mode='lines',
        line={'color':color,'width':2.5,'dash':dash},customdata=hover,
        hovertemplate='%{x|%H:%M}<br>Speed: %{y:.2f}<br>Rekaman: %{customdata[0]:,}<br>Bus: %{customdata[1]:,}<extra>%{fullData.name}</extra>'))
fig1.update_layout(title='1. Perubahan speed selama periode pengamatan',xaxis_title='26 November 2019 — waktu pada data',
    yaxis_title='Speed (satuan dataset)',hovermode='x unified',legend={'orientation':'h','x':0,'y':1.02,'yanchor':'bottom'})
fig1.update_xaxes(tickformat='%H:%M',rangeslider={'visible':True,'thickness':0.16})
publish(fig1,'01_tren_waktu.html',560)

### 2. Distribusi speed dan null
Histogram berinterval mencakup semua rekaman. Nol dipisahkan dari speed rendah
positif dan dari null. Persentase grafik memakai seluruh baris; statistik speed
memakai nilai yang diketahui. Nol dapat berarti berhenti di halte, depo, atau
antrean, sehingga tidak sendirinya membuktikan kemacetan.

In [18]:
fig2=go.Figure(go.Bar(x=distribution['kelompok_speed'].to_list(),y=distribution['jumlah_rekaman'].to_list(),
    marker_color=[COLORS['orange'] if v=='Tidak diketahui' else COLORS['navy'] if v=='0' else COLORS['teal'] for v in distribution['kelompok_speed']],
    customdata=distribution['persen_seluruh_rekaman'].to_list(),
    text=[f'{v:.2f}%' for v in distribution['persen_seluruh_rekaman']],textposition='outside',cliponaxis=False,
    hovertemplate='Speed: %{x}<br>Rekaman: %{y:,}<br>Proporsi semua rekaman: %{customdata:.3f}%<extra></extra>'))
fig2.update_layout(title='2. Banyaknya rekaman pada setiap rentang speed',xaxis_title='Rentang speed (satuan dataset)',yaxis_title='Jumlah rekaman GPS')
fig2.update_yaxes(range=[0,distribution['jumlah_rekaman'].max()*1.16])
publish(fig2,'02_distribusi_speed.html')

### 3. Hubungan jumlah rekaman dan speed per bus
Setiap titik adalah satu bus dengan minimal 30 rekaman. Warna menunjukkan median
jeda pelaporan. Korelasi bersifat deskriptif; durasi tercatat, koridor, serta
frekuensi GPS dapat memengaruhi hubungan. Tidak dilakukan inferensi kausal atau
uji signifikansi statistik terhadap pengamatan berulang ini.

In [19]:
fig3=px.scatter(scatter_bus,x='jumlah_rekaman',y='speed_mean',color='median_gap_detik',color_continuous_scale='Cividis',
    hover_name='bus_code',hover_data={'jumlah_rekaman':':,','speed_mean':':.2f','median_gap_detik':':.1f','persen_speed_nol':':.2f'},
    labels={'jumlah_rekaman':'Jumlah rekaman per bus','speed_mean':'Rata-rata speed per bus','median_gap_detik':'Median jeda (detik)','persen_speed_nol':'Speed nol (%)'},
    title=f'3. Cakupan pelaporan dan speed per bus | Pearson r = {pearson_r:.3f}')
fig3.update_traces(marker={'size':7,'opacity':0.6})
publish(fig3,'03_hubungan_pelaporan_speed.html')

### 4. Komposisi kode koridor
Ditampilkan 12 kode dengan rekaman terbanyak, selain kode 0. Warna menunjukkan
rata-rata speed per bus pada kode tersebut. Banyak rekaman bukan ukuran
permintaan penumpang. Bus dapat muncul di beberapa kode, sehingga hitungan bus
unik antar-koridor tidak boleh langsung dijumlahkan.

In [20]:
fig4=px.bar(top_corridors,x='jumlah_rekaman',y='corridor',orientation='h',color='speed_mean_bus',color_continuous_scale='Cividis',
    hover_data={'bus_unik':True,'speed_mean_bus':':.2f','jumlah_rekaman':':,'},
    labels={'corridor':'Kode koridor','jumlah_rekaman':'Jumlah rekaman GPS','speed_mean_bus':'Speed per bus','bus_unik':'Bus unik tercatat'},
    title='4. Dua belas kode koridor dengan rekaman terbanyak')
fig4.update_yaxes(type='category',categoryorder='array',categoryarray=list(reversed(top_corridors['corridor'].to_list())))
fig4.update_traces(text=top_corridors['jumlah_rekaman'].to_list(),texttemplate='%{text:,}',textposition='outside',cliponaxis=False)
fig4.update_xaxes(range=[0,top_corridors['jumlah_rekaman'].max()*1.12])
publish(fig4,'04_komposisi_koridor.html',600)

### 5. Perbandingan koridor per jam
Heatmap membandingkan delapan kode teratas pada empat jam. Setiap sel memakai
rata-rata speed per bus pada kode dan jam yang sama; sel di bawah 10 bus dengan
speed valid tidak ditampilkan. Ini perbandingan kelompok, bukan pola musiman.

In [21]:
hours=sorted(hourly['jam'].to_list())
heatmap_rows=corridor_hour.filter(pl.col('corridor').is_in(top_codes))
lookup={(r['corridor'],r['jam']):r for r in heatmap_rows.iter_rows(named=True)}
heat_z=[];heat_text=[]
for corridor in top_codes:
    z=[];texts=[]
    for hour in hours:
        r=lookup.get((corridor,hour))
        z.append(r['speed_mean_bus'] if r and r['bus_dengan_speed_valid']>=10 else None)
        texts.append(f"Kode {corridor} | {hour:%H:%M}<br>Bus dengan speed valid: {r['bus_dengan_speed_valid']}<br>Rekaman: {r['jumlah_rekaman']:,}" if r else 'Data tidak tersedia')
    heat_z.append(z);heat_text.append(texts)
fig5=go.Figure(go.Heatmap(z=heat_z,x=[f'{h:%H}:00–{h:%H}:59' for h in hours],y=top_codes,text=heat_text,
    texttemplate='%{z:.2f}',textfont={'size':12},colorscale='Cividis',colorbar={'title':'Speed per bus'},
    hovertemplate='%{text}<br>Speed per bus: %{z:.2f}<extra></extra>'))
fig5.update_layout(title='5. Perbedaan speed per koridor dan jam pengamatan',xaxis_title='Jam pada 26 November 2019',yaxis_title='Kode koridor')
fig5.update_yaxes(type='category',autorange='reversed')
publish(fig5,'05_koridor_per_jam.html')

### 6. Pola spasial posisi GPS
Titik mewakili pusat sel koordinat, bukan halte atau ruas resmi. Warna adalah
median speed; ukuran mengikuti log jumlah rekaman agar sel padat tidak menutupi
sel lain. Tidak ada basemap eksternal. Sel padat rekaman bisa berarti halte,
depo, antrean, jalur yang sering dilewati, atau frekuensi GPS yang lebih tinggi.

In [22]:
spatial_for_chart=spatial_plot.with_columns((pl.col('jumlah_rekaman').cast(pl.Float64)+1).log().alias('ukuran_titik'))
fig6=px.scatter(spatial_for_chart,x='longitude',y='latitude',size='ukuran_titik',size_max=14,color='speed_median',color_continuous_scale='Cividis',
    hover_data={'jumlah_rekaman':':,','bus_unik':True,'speed_median':':.2f','longitude':':.4f','latitude':':.4f','ukuran_titik':False},
    labels={'longitude':'Bujur','latitude':'Lintang','speed_median':'Median speed','jumlah_rekaman':'Rekaman dalam sel','bus_unik':'Bus unik dalam sel'},
    title='6. Sebaran rekaman GPS dan median speed per sel koordinat')
fig6.update_traces(marker={'opacity':0.75,'line':{'width':0.35,'color':'#FFFFFF'}})
fig6.update_yaxes(scaleanchor='x',scaleratio=1,tickformat='.2f');fig6.update_xaxes(tickformat='.2f')
publish(fig6,'06_pola_spasial.html',650)

## Bagian 4: Enam Insight Analitik
Angka dihitung dari hasil di atas. Temuan dipisahkan dari batas interpretasi
dan rekomendasi agar analisis deskriptif tidak berubah menjadi klaim sebab akibat.

In [23]:
def num(v,d=0):return f'{v:,.{d}f}'.replace(',','_').replace('.',',').replace('_','.')
slow_hour=hourly.sort('speed_mean_bus').row(0,named=True)
fast_hour=hourly.sort('speed_mean_bus',descending=True).row(0,named=True)
first_hour=hourly.row(0,named=True);last_hour=hourly.row(-1,named=True)
code_zero=corridor_stats.filter(pl.col('corridor')=='0')['jumlah_rekaman'].sum()
top=top_corridors.row(0,named=True);grid_top=spatial_plot.row(0,named=True)
eligible=heatmap_rows.filter(pl.col('bus_dengan_speed_valid')>=10)
slow=eligible.sort(['speed_mean_bus','corridor','jam']).row(0,named=True)
fast=eligible.sort(['speed_mean_bus','corridor','jam'],descending=[True,False,False]).row(0,named=True)
insights=[
 {'judul':'Tren waktu berubah, tetapi belum membuktikan pola musiman',
  'temuan':f"Rata-rata speed per bus per jam terendah pada {slow_hour['jam']:%H}:00 sebesar {num(slow_hour['speed_mean_bus'],2)}, tertinggi pada {fast_hour['jam']:%H}:00 sebesar {num(fast_hour['speed_mean_bus'],2)}. Bus tercatat berubah dari {num(first_hour['bus_tercatat'])} pada jam pertama menjadi {num(last_hour['bus_tercatat'])} pada jam terakhir (grafik 1 dan tabel per jam).",
  'batasan':'Pengamatan hanya empat jam pada satu tanggal; tidak dapat digeneralisasikan menjadi pola jam sibuk atau pola harian berulang.',
  'rekomendasi':'Kumpulkan beberapa hari dan bandingkan jam yang sama sebelum mengubah jadwal layanan.'},
 {'judul':'Speed rendah mendominasi dan distribusi tidak simetris',
  'temuan':f"Dari {num(valid_speed)} speed diketahui, {num(zero_speed/valid_speed*100,2)}% bernilai nol dan {num(low_speed/valid_speed*100,2)}% paling tinggi 5. Mean {num(df['speed'].mean(),2)} melebihi median {num(df['speed'].median(),2)}. Ada {num(null_speed)} speed null yang tidak diperlakukan sebagai nol (grafik 2).",
  'batasan':'Speed nol tidak membedakan halte, depo, antrean, dan keadaan lalu lintas. Proporsi rekaman belum berbobot durasi.',
  'rekomendasi':'Gunakan mean bersama median; cocokkan titik berhenti dengan halte/depo dan jeda waktu GPS.'},
 {'judul':'Cakupan pelaporan perlu diperhitungkan saat membandingkan bus',
  'temuan':f"Pada {num(scatter_bus.height)} bus dengan minimal 30 rekaman, Pearson r antara jumlah rekaman dan rata-rata speed adalah {num(pearson_r,3)}. Rekaman per bus pada data lengkap berkisar {num(bus_stats['jumlah_rekaman'].min())}–{num(bus_stats['jumlah_rekaman'].max())}, dengan median {num(bus_stats['jumlah_rekaman'].median())} (grafik 3).",
  'batasan':'Korelasi deskriptif dapat dipengaruhi durasi tercatat, jadwal, koridor, dan frekuensi GPS; tidak dilakukan inferensi kausal atau uji signifikansi.',
  'rekomendasi':'Bandingkan bus pada jendela waktu yang sama dan audit bus dengan pelaporan sangat sedikit.'},
 {'judul':'Komposisi rekaman terkonsentrasi pada beberapa kode koridor',
  'temuan':f"Selain kode 0, kode {top['corridor']} mempunyai rekaman terbanyak: {num(top['jumlah_rekaman'])} dari {num(top['bus_unik'])} bus unik. Kode 0 mencakup {num(code_zero)} rekaman ({num(code_zero/df.height*100,2)}%) dan dipisahkan dari pemeringkatan (grafik 4).",
  'batasan':'Banyak rekaman bukan jumlah penumpang. Arti kode 0 belum tersedia dalam metadata lokal.',
  'rekomendasi':'Verifikasi kamus koridor dan lengkapi data penumpang serta jadwal sebelum memutuskan penambahan armada.'},
 {'judul':'Perbandingan koridor bergantung pada jam pengamatan',
  'temuan':f"Di sel heatmap dengan minimal 10 bus ber-speed valid, nilai terendah terjadi pada kode {slow['corridor']} pukul {slow['jam']:%H}:00 ({num(slow['speed_mean_bus'],2)}; {num(slow['bus_dengan_speed_valid'])} bus). Tertinggi pada kode {fast['corridor']} pukul {fast['jam']:%H}:00 ({num(fast['speed_mean_bus'],2)}; {num(fast['bus_dengan_speed_valid'])} bus) (grafik 5).",
  'batasan':'Perbedaan dapat berasal dari rute, waktu berhenti, dan komposisi bus; speed rendah belum membuktikan koridor berkinerja buruk.',
  'rekomendasi':'Periksa pasangan koridor–jam bernilai rendah bersama kondisi halte dan perjalanan aktual.'},
 {'judul':'Pola spasial mengarahkan pemeriksaan lokasi',
  'temuan':f"Sel terpadat berpusat pada bujur {grid_top['longitude']:.4f}, lintang {grid_top['latitude']:.4f}, berisi {num(grid_top['jumlah_rekaman'])} rekaman dari {num(grid_top['bus_unik'])} bus dengan median speed {num(grid_top['speed_median'],2)}. Grafik menampilkan {num(spatial_plot.height)} sel; {num(outside_count)} rekaman bertanda luar wilayah M2 diaudit terpisah (grafik 6).",
  'batasan':'Sel koordinat bukan ruas jalan atau halte resmi; kepadatan rekaman dipengaruhi frekuensi GPS dan bus yang berhenti lama.',
  'rekomendasi':'Cocokkan sel dengan jalan, halte, dan depo; periksa dua titik luar wilayah sebelum mengambil keputusan operasional.'},
]
insight_markdown='\n\n'.join(f"### {i}. {x['judul']}\n\n**Temuan:** {x['temuan']}\n\n**Batas interpretasi:** {x['batasan']}\n\n**Rekomendasi:** {x['rekomendasi']}" for i,x in enumerate(insights,1))
display(Markdown(insight_markdown))
(OUTPUT_DIR/'insights.md').write_text(insight_markdown+'\n',encoding='utf-8')

### 1. Tren waktu berubah, tetapi belum membuktikan pola musiman

**Temuan:** Rata-rata speed per bus per jam terendah pada 14:00 sebesar 9,99, tertinggi pada 16:00 sebesar 11,12. Bus tercatat berubah dari 1.674 pada jam pertama menjadi 1.723 pada jam terakhir (grafik 1 dan tabel per jam).

**Batas interpretasi:** Pengamatan hanya empat jam pada satu tanggal; tidak dapat digeneralisasikan menjadi pola jam sibuk atau pola harian berulang.

**Rekomendasi:** Kumpulkan beberapa hari dan bandingkan jam yang sama sebelum mengubah jadwal layanan.

### 2. Speed rendah mendominasi dan distribusi tidak simetris

**Temuan:** Dari 726.975 speed diketahui, 37,78% bernilai nol dan 56,91% paling tinggi 5. Mean 10,73 melebihi median 2,10. Ada 99 speed null yang tidak diperlakukan sebagai nol (grafik 2).

**Batas interpretasi:** Speed nol tidak membedakan halte, depo, antrean, dan keadaan lalu lintas. Proporsi rekaman belum berbobot durasi.

**Rekomendasi:** Gunakan mean bersama median; cocokkan titik berhenti dengan halte/depo dan jeda waktu GPS.

### 3. Cakupan pelaporan perlu diperhitungkan saat membandingkan bus

**Temuan:** Pada 1.759 bus dengan minimal 30 rekaman, Pearson r antara jumlah rekaman dan rata-rata speed adalah 0,100. Rekaman per bus pada data lengkap berkisar 1–453, dengan median 444 (grafik 3).

**Batas interpretasi:** Korelasi deskriptif dapat dipengaruhi durasi tercatat, jadwal, koridor, dan frekuensi GPS; tidak dilakukan inferensi kausal atau uji signifikansi.

**Rekomendasi:** Bandingkan bus pada jendela waktu yang sama dan audit bus dengan pelaporan sangat sedikit.

### 4. Komposisi rekaman terkonsentrasi pada beberapa kode koridor

**Temuan:** Selain kode 0, kode 1 mempunyai rekaman terbanyak: 46.915 dari 174 bus unik. Kode 0 mencakup 90.375 rekaman (12,43%) dan dipisahkan dari pemeringkatan (grafik 4).

**Batas interpretasi:** Banyak rekaman bukan jumlah penumpang. Arti kode 0 belum tersedia dalam metadata lokal.

**Rekomendasi:** Verifikasi kamus koridor dan lengkapi data penumpang serta jadwal sebelum memutuskan penambahan armada.

### 5. Perbandingan koridor bergantung pada jam pengamatan

**Temuan:** Di sel heatmap dengan minimal 10 bus ber-speed valid, nilai terendah terjadi pada kode 4 pukul 15:00 (10,11; 91 bus). Tertinggi pada kode 10 pukul 16:00 (13,92; 76 bus) (grafik 5).

**Batas interpretasi:** Perbedaan dapat berasal dari rute, waktu berhenti, dan komposisi bus; speed rendah belum membuktikan koridor berkinerja buruk.

**Rekomendasi:** Periksa pasangan koridor–jam bernilai rendah bersama kondisi halte dan perjalanan aktual.

### 6. Pola spasial mengarahkan pemeriksaan lokasi

**Temuan:** Sel terpadat berpusat pada bujur 106.8225, lintang -6.1675, berisi 13.029 rekaman dari 365 bus dengan median speed 0,70. Grafik menampilkan 1.061 sel; 2 rekaman bertanda luar wilayah M2 diaudit terpisah (grafik 6).

**Batas interpretasi:** Sel koordinat bukan ruas jalan atau halte resmi; kepadatan rekaman dipengaruhi frekuensi GPS dan bus yang berhenti lama.

**Rekomendasi:** Cocokkan sel dengan jalan, halte, dan depo; periksa dua titik luar wilayah sebelum mengambil keputusan operasional.

3152

## Bagian 5: Ekspor dan Pemeriksaan Akhir

Input mentah aktual 72.981.601 byte dan 775.458 baris belum memenuhi syarat
minimal 500 MB atau lebih dari satu juta baris. Data tidak digandakan untuk
memperbesar ukuran. Cakupan periode, satuan speed yang belum diverifikasi,
dan template cleaning M2 perlu diperhatikan sebelum pengumpulan final.

In [24]:
assert len(figures)==6 and len(insights)>=5
assert hashlib.sha256(DATA_PATH.read_bytes()).hexdigest()==INPUT_SHA256
raw_path=PROJECT_ROOT/'data/raw/transjakarta_gps.csv'
raw_hash_matches=None
if raw_path.is_file():
    raw_hash_matches=hashlib.sha256(raw_path.read_bytes()).hexdigest()==audit_m2['sha256_raw']
    assert raw_hash_matches
validation=pl.DataFrame([
 {'pemeriksaan':'Baris sesuai audit M2','lulus':df.height==audit_m2['baris_bersih']},
 {'pemeriksaan':'Seluruh rekaman masuk agregasi temporal','lulus':temporal['jumlah_rekaman'].sum()==df.height},
 {'pemeriksaan':'Distribusi mencakup seluruh nilai termasuk null','lulus':distribution['jumlah_rekaman'].sum()==df.height},
 {'pemeriksaan':'Grid ditambah audit mencakup seluruh posisi','lulus':spatial_grid['jumlah_rekaman'].sum()+spatial_audit.height==df.height},
 {'pemeriksaan':'MA Polars cocok dengan DuckDB','lulus':ma_error<1e-9},
 {'pemeriksaan':'Lag Polars cocok dengan DuckDB','lulus':lag_error<1e-9},
 {'pemeriksaan':'Enam HTML interaktif tersedia','lulus':len(figures)==6 and all((FIGURE_DIR/p).is_file() for p in figure_files)},
 {'pemeriksaan':'Minimal lima insight','lulus':len(insights)>=5},
 {'pemeriksaan':'Input bersih tidak berubah','lulus':hashlib.sha256(DATA_PATH.read_bytes()).hexdigest()==INPUT_SHA256},
])
assert validation['lulus'].all()
validation.write_csv(OUTPUT_DIR/'validasi_milestone3.csv')
summary={'nama':'Dino Ibnu Zein','nim':'202310370311107','github':'dinoibnuzein',
 'input':DATA_PATH.relative_to(PROJECT_ROOT).as_posix(),'sha256_input_bersih':INPUT_SHA256,
 'sha256_raw_sesuai_audit':raw_hash_matches,'baris_raw':audit_m2['baris_awal'],'ukuran_raw_bytes':audit_m2['ukuran_bytes'],
 'skala_tugas_terpenuhi':audit_m2.get('skala_memenuhi',False),'baris_bersih':df.height,'bus_unik':df['bus_code'].n_unique(),
 'kode_koridor_unik':df['corridor'].n_unique(),'periode_awal':start_time.isoformat(),'periode_akhir':end_time.isoformat(),
 'speed_mean':df['speed'].mean(),'speed_median':df['speed'].median(),'speed_null':null_speed,
 'speed_nol_persen_dari_valid':zero_speed/valid_speed*100,'speed_le5_persen_dari_valid':low_speed/valid_speed*100,
 'pearson_jumlah_rekaman_speed_per_bus':pearson_r,'bus_pada_scatter':scatter_bus.height,
 'jumlah_interval_5_menit':temporal.height,'jumlah_grafik_interaktif':6,'jumlah_insight':6,
 'sel_spasial_ditampilkan':spatial_plot.height,'titik_di_luar_wilayah_pemeriksaan_m2':outside_count,'versi':versions,'insights':insights}
(OUTPUT_DIR/'ringkasan_analisis.json').write_text(json.dumps(summary,ensure_ascii=False,indent=2),encoding='utf-8')
sections=[]
for i,fig in enumerate(figures):
    chart=fig.to_html(full_html=False,include_plotlyjs=(i==0),config=CONFIG,div_id=f'milestone3_chart_{i+1}')
    x=insights[i]
    sections.append('<section class="card">'+chart+'<div class="reading"><h3>'+html.escape(x['judul'])+'</h3><p>'+html.escape(x['temuan'])+'</p><p class="note">'+html.escape(x['batasan'])+'</p></div></section>')
report=("""<!doctype html><html lang="id"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1"><title>Milestone 3 — Transjakarta GPS</title><style>
body{margin:0;background:#eef3f7;color:#21364b;font:15px/1.65 Arial,sans-serif}main{max-width:1200px;margin:auto;padding:30px 18px}h1{font-size:32px;line-height:1.2}.card{background:white;border:1px solid #dce5ed;border-radius:12px;margin:20px 0;overflow:hidden}.reading,.intro{padding:18px 26px}.reading{border-top:1px solid #e8eef3}.note{color:#5b6e80;font-size:13px}.kpis{display:flex;flex-wrap:wrap;gap:12px}.kpi{background:#15324f;color:white;border-radius:10px;padding:14px 24px;flex:1;min-width:170px}.kpi strong{display:block;font-size:27px}.kpi span{font-size:12px}</style></head><body><main><p>ANALISIS BIG DATA / MILESTONE 3</p><h1>Empat jam rekaman GPS Transjakarta</h1><p>Dino Ibnu Zein · 202310370311107 · dinoibnuzein</p>"""
    +'<div class="kpis"><div class="kpi"><strong>'+num(df.height)+'</strong><span>rekaman bersih</span></div><div class="kpi"><strong>'+num(df['bus_code'].n_unique())+'</strong><span>bus unik tercatat</span></div><div class="kpi"><strong>6 grafik</strong><span>interaktif dan dapat dibuka tanpa internet</span></div></div>'
    +'<section class="card intro"><h3>Cakupan dan cara membaca</h3><p>26 November 2019, 14.00–17.59. Hover untuk membaca nilai; zoom untuk memperbesar; klik legenda untuk memilih seri. Speed mengikuti satuan dataset. Jumlah rekaman bukan jumlah penumpang, dan speed nol bukan bukti tunggal kemacetan.</p><p class="note">File raw aktual 72,98 MB dan 775.458 baris, belum memenuhi syarat skala tugas. Notebook M2 di arsip masih template; M3 menggunakan Parquet bersih dan audit yang tersedia.</p></section>'
    +''.join(sections)+'<section class="card intro"><h3>Provenance dan penggunaan AI</h3><p>Angka agregasi dan checksum tersedia pada tabel CSV dan ringkasan_analisis.json. ChatGPT digunakan dalam penyusunan kode, visualisasi, dan analisis Milestone 3. Hasilnya telah diperiksa melalui eksekusi notebook dan pengecekan konsistensi angka.</p></section></main></body></html>')
(OUTPUT_DIR/'ringkasan_interaktif.html').write_text(report,encoding='utf-8')
con.close()
display(validation)
print('Selesai: enam grafik interaktif, enam insight, tabel agregasi, dan HTML mandiri.')

pemeriksaan,lulus
str,bool
"""Baris sesuai audit M2""",true
"""Seluruh rekaman masuk agregasi…",true
"""Distribusi mencakup seluruh ni…",true
"""Grid ditambah audit mencakup s…",true
"""MA Polars cocok dengan DuckDB""",true
"""Lag Polars cocok dengan DuckDB""",true
"""Enam HTML interaktif tersedia""",true
"""Minimal lima insight""",true
"""Input bersih tidak berubah""",true


Selesai: enam grafik interaktif, enam insight, tabel agregasi, dan HTML mandiri.


## Metode, Sumber dan AI Disclosure

- Sumber angka adalah Parquet hasil M2 dan audit cleaning dari arsip proyek.
- Halaman dataset yang dicatat dalam proyek:
  https://www.kaggle.com/datasets/rasyidstat/transjakarta-bus-gps-data.
- Referensi API:
  https://docs.pola.rs/api/python/stable/reference/dataframe/api/polars.DataFrame.group_by_dynamic.html,
  https://duckdb.org/docs/stable/sql/functions/window_functions,
  https://plotly.com/python/interactive-html-export/.
- Rata-rata per bus merupakan penyeimbangan bobot bus, bukan kecepatan
  perjalanan yang sudah berbobot jarak atau waktu. Pengamatan pada bus yang
  sama berulang, sehingga korelasi dilaporkan tanpa klaim sebab akibat.
- ChatGPT digunakan dalam penyusunan kode, visualisasi, dan analisis Milestone 3. Hasilnya telah diperiksa melalui eksekusi notebook dan pengecekan   konsistensi angka.
